# Phase 0: Setup & Data Validation

This phase focuses on environment setup verification, global seed initialization, dataset file checks, and rigorous data structure validation. In accordance with the project constraints, no models are defined, trained, or evaluated in this section, and no plots are generated.

In [1]:
import sys
import random
import numpy as np
import sklearn
import xgboost

# 1. Report runtime versions
print(f"Python version: {sys.version}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"xgboost version: {xgboost.__version__}")

try:
    import tensorflow as tf
    tf_version = tf.__version__
except ImportError:
    tf_version = "not installed"
print(f"tensorflow version: {tf_version}")

try:
    import scikeras
    scikeras_version = scikeras.__version__
except ImportError:
    scikeras_version = "not installed"
print(f"scikeras version: {scikeras_version}")

# 2. Set global seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if tf_version != "not installed":
    tf.random.set_seed(SEED)
    print(f"\nGlobal seed set to {SEED} (random, numpy, tensorflow)")
else:
    print(f"\nGlobal seed set to {SEED} (random, numpy)")

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
scikit-learn version: 1.8.0
xgboost version: 3.2.0
tensorflow version: 2.21.0
scikeras version: 0.13.0

Global seed set to 42 (random, numpy, tensorflow)


## Data Listing & Pairing

We list all CSV files inside the `./data` directory, print their absolute paths, and group them into three dataset pairs: `chantest`, `nanion`, and `manual`.

In [2]:
import os

data_dir = "./data"
all_files = os.listdir(data_dir)
csv_files = [f for f in all_files if f.endswith(".csv")]

print("CSV Files Found in ./data:")
for f in sorted(csv_files):
    abs_path = os.path.abspath(os.path.join(data_dir, f))
    print(f"  - Filename: {f}")
    print(f"    Absolute Path: {abs_path.replace(chr(92), '/')}")

# Define expected pairs
pairs = {
    "chantest": {"train": "chantest_training.csv", "test": "chantest_testing.csv"},
    "nanion": {"train": "nanion_training.csv", "test": "nanion_testing.csv"},
    "manual": {"train": "manual_training(in).csv", "test": "manual_testing(in).csv"}
}

print("\nGrouped Dataset Pairs:")
for pair_name, files in pairs.items():
    print(f"\nPair: {pair_name}")
    train_f = files["train"]
    if train_f in csv_files:
        print(f"  - Train file: {train_f} (Found)")
    else:
        print(f"  - Train file: {train_f} (MISSING)")
    test_f = files["test"]
    if test_f in csv_files:
        print(f"  - Test file:  {test_f} (Found)")
    else:
        print(f"  - Test file:  {test_f} (MISSING)")

CSV Files Found in ./data:
  - Filename: chantest_testing.csv
    Absolute Path: D:/1. COLLEGE/1. HERE WE GO/8. TUGAS AKHIR/1. RISET ML/5. Here We Go Bismillah/TrainAI/data/chantest_testing.csv
  - Filename: chantest_training.csv
    Absolute Path: D:/1. COLLEGE/1. HERE WE GO/8. TUGAS AKHIR/1. RISET ML/5. Here We Go Bismillah/TrainAI/data/chantest_training.csv
  - Filename: manual_testing(in).csv
    Absolute Path: D:/1. COLLEGE/1. HERE WE GO/8. TUGAS AKHIR/1. RISET ML/5. Here We Go Bismillah/TrainAI/data/manual_testing(in).csv
  - Filename: manual_training(in).csv
    Absolute Path: D:/1. COLLEGE/1. HERE WE GO/8. TUGAS AKHIR/1. RISET ML/5. Here We Go Bismillah/TrainAI/data/manual_training(in).csv
  - Filename: nanion_testing.csv
    Absolute Path: D:/1. COLLEGE/1. HERE WE GO/8. TUGAS AKHIR/1. RISET ML/5. Here We Go Bismillah/TrainAI/data/nanion_testing.csv
  - Filename: nanion_training.csv
    Absolute Path: D:/1. COLLEGE/1. HERE WE GO/8. TUGAS AKHIR/1. RISET ML/5. Here We Go Bismilla

## Data Loading

We load each CSV file into a dictionary of pandas DataFrames keyed by the pattern `<pair_name>_<split>`.

In [3]:
import pandas as pd

datasets = {}
for pair_name, files in pairs.items():
    for split in ["train", "test"]:
        file_name = files[split]
        file_path = os.path.join(data_dir, file_name)
        if os.path.exists(file_path):
            key = f"{pair_name}_{split}"
            datasets[key] = pd.read_csv(file_path)
            print(f"Loaded '{file_name}' into dictionary as key '{key}' with shape {datasets[key].shape}")
        else:
            print(f"ERROR: File '{file_name}' not found.")

Loaded 'chantest_training.csv' into dictionary as key 'chantest_train' with shape (24000, 14)
Loaded 'chantest_testing.csv' into dictionary as key 'chantest_test' with shape (32000, 14)
Loaded 'nanion_training.csv' into dictionary as key 'nanion_train' with shape (24000, 14)
Loaded 'nanion_testing.csv' into dictionary as key 'nanion_test' with shape (32000, 14)
Loaded 'manual_training(in).csv' into dictionary as key 'manual_train' with shape (24000, 14)
Loaded 'manual_testing(in).csv' into dictionary as key 'manual_test' with shape (32000, 14)


## Dataset Pairs Validation Reports

For each dataset pair, we run comprehensive checks on:
1. Train/Test row counts (expected 24,000 / 32,000; flags mismatch but does not crash).
2. Presence of the 11 canonical biomarker columns (`qNet`, `dvdtmax`, `vmax`, `vrest`, `APD50`, `APD90`, `max_dv`, `camax`, `carest`, `CaTD50`, `CaTD90`).
3. Label column name and per-class value counts.
4. Drug-name column, unique drug counts (expected 12 in train, 16 in test), and sample count per drug (expected 2,000).
5. Disjoint drug sets (confirming zero drug overlap between train and test).
6. Non-numeric or NaN values in the biomarker columns.

In [4]:
biomarkers = [
    'qNet', 'dvdtmax', 'vmax', 'vrest', 'APD50', 'APD90', 'max_dv', 'camax', 'carest', 'CaTD50', 'CaTD90'
]

pair_status = {}

for pair_name in ["chantest", "nanion", "manual"]:
    print("\n" + "="*60)
    print(f"VALIDATION REPORT FOR PAIR: {pair_name.upper()}")
    print("="*60)
    
    train_key = f"{pair_name}_train"
    test_key = f"{pair_name}_test"
    
    if train_key not in datasets or test_key not in datasets:
        print("ERROR: Datasets for this pair are not loaded. Skipping.")
        pair_status[pair_name] = "FAIL"
        continue
        
    df_train = datasets[train_key]
    df_test = datasets[test_key]
    
    checks_passed = True
    
    # 1. Row counts check
    print("--- 1. Row Counts ---")
    train_rows = len(df_train)
    test_rows = len(df_test)
    print(f"Train row count: {train_rows:,} (Expected: 24,000)")
    print(f"Test row count:  {test_rows:,} (Expected: 32,000)")
    if train_rows == 24000 and test_rows == 32000:
        row_check = "PASS"
        print(f"Row count check: {row_check}")
    else:
        row_check = "FLAG"
        print(f"Row count check: {row_check} (WARNING: Row counts differ from chantest reference)")
        # Note: Do not crash
        
    # 2. Biomarkers presence check
    print("\n--- 2. Biomarker Columns Presence ---")
    missing_train = [c for c in biomarkers if c not in df_train.columns]
    missing_test = [c for c in biomarkers if c not in df_test.columns]
    if not missing_train and not missing_test:
        biomarkers_present_check = "PASS"
        print("All 11 biomarker columns are present in both splits.")
    else:
        biomarkers_present_check = "FAIL"
        checks_passed = False
        if missing_train:
            print(f"  Missing in train: {missing_train}")
        if missing_test:
            print(f"  Missing in test:  {missing_test}")
    print(f"Biomarker columns presence: {biomarkers_present_check}")
    
    # 3. Label column checks
    print("\n--- 3. Label Column & Class Counts ---")
    label_col = 'label'
    print(f"Label column detected: '{label_col}'")
    print("Train label value counts:")
    train_label_counts = df_train[label_col].value_counts()
    for val, cnt in train_label_counts.items():
        print(f"  - {val}: {cnt}")
    print("Test label value counts:")
    test_label_counts = df_test[label_col].value_counts()
    for val, cnt in test_label_counts.items():
        print(f"  - {val}: {cnt}")
    
    expected_classes = {'high', 'intermediate', 'low'}
    train_classes = set(df_train[label_col].unique())
    test_classes = set(df_test[label_col].unique())
    if train_classes == expected_classes and test_classes == expected_classes:
        label_check = "PASS"
    else:
        label_check = "FAIL"
        checks_passed = False
        print(f"  Unexpected classes: Train {train_classes}, Test {test_classes}")
    print(f"Label column check: {label_check}")
    
    # 4. Drug checks
    print("\n--- 4. Drug Columns & Sample Counts ---")
    drug_col = 'drug_name'
    print(f"Drug column detected: '{drug_col}'")
    train_drugs = df_train[drug_col].unique()
    test_drugs = df_test[drug_col].unique()
    print(f"Unique drugs in train ({len(train_drugs)}): {sorted(list(train_drugs))}")
    print(f"Unique drugs in test  ({len(test_drugs)}): {sorted(list(test_drugs))}")
    
    train_drug_counts = df_train[drug_col].value_counts()
    test_drug_counts = df_test[drug_col].value_counts()
    train_min_count = train_drug_counts.min()
    train_max_count = train_drug_counts.max()
    test_min_count = test_drug_counts.min()
    test_max_count = test_drug_counts.max()
    print(f"Train per-drug sample counts: Min={train_min_count}, Max={train_max_count} (Expected: 2000)")
    print(f"Test per-drug sample counts:  Min={test_min_count}, Max={test_max_count} (Expected: 2000)")
    
    if len(train_drugs) == 12 and len(test_drugs) == 16 and train_min_count == 2000 and train_max_count == 2000 and test_min_count == 2000 and test_max_count == 2000:
        drug_check = "PASS"
    else:
        drug_check = "FAIL"
        checks_passed = False
        print("  Drug counts or drug sample size differs from spec.")
    print(f"Drug check: {drug_check}")
    
    # 5. Disjoint check
    print("\n--- 5. Drug Sets Disjoint Verification ---")
    overlap = set(train_drugs).intersection(set(test_drugs))
    if not overlap:
        disjoint_check = "PASS"
        print("Train and test drug sets are disjoint (zero overlap).")
    else:
        disjoint_check = "FAIL"
        checks_passed = False
        print(f"  Overlapping drugs found: {overlap}")
    print(f"Disjoint drug sets check: {disjoint_check}")
    
    # 6. Numeric and NaNs check
    print("\n--- 6. Biomarker Numeric & NaN Checks ---")
    nans_train = df_train[biomarkers].isna().sum()
    nans_test = df_test[biomarkers].isna().sum()
    total_nans_train = nans_train.sum()
    total_nans_test = nans_test.sum()
    
    non_numeric_train = 0
    non_numeric_test = 0
    for col in biomarkers:
        if col in df_train.columns:
            non_numeric_train += pd.to_numeric(df_train[col], errors='coerce').isna().sum() - df_train[col].isna().sum()
        if col in df_test.columns:
            non_numeric_test += pd.to_numeric(df_test[col], errors='coerce').isna().sum() - df_test[col].isna().sum()
            
    print(f"Train NaNs in biomarkers: {total_nans_train}")
    if total_nans_train > 0:
        print("  Train NaNs by column:")
        print(nans_train[nans_train > 0])
    print(f"Train non-numeric values in biomarkers: {non_numeric_train}")
    
    print(f"Test NaNs in biomarkers: {total_nans_test}")
    if total_nans_test > 0:
        print("  Test NaNs by column:")
        print(nans_test[nans_test > 0])
    print(f"Test non-numeric values in biomarkers: {non_numeric_test}")
    
    if total_nans_train == 0 and total_nans_test == 0 and non_numeric_train == 0 and non_numeric_test == 0:
        numeric_check = "PASS"
    else:
        numeric_check = "FAIL"
        checks_passed = False
        print("  Biomarker columns contain NaN or non-numeric values.")
    print(f"Numeric/NaN checks: {numeric_check}")
    
    if checks_passed:
        pair_status[pair_name] = "PASS"
    else:
        pair_status[pair_name] = "FAIL"


VALIDATION REPORT FOR PAIR: CHANTEST
--- 1. Row Counts ---
Train row count: 24,000 (Expected: 24,000)
Test row count:  32,000 (Expected: 32,000)
Row count check: PASS

--- 2. Biomarker Columns Presence ---
All 11 biomarker columns are present in both splits.
Biomarker columns presence: PASS

--- 3. Label Column & Class Counts ---
Label column detected: 'label'
Train label value counts:
  - high: 8000
  - intermediate: 8000
  - low: 8000
Test label value counts:
  - intermediate: 14000
  - low: 10000
  - high: 8000
Label column check: PASS

--- 4. Drug Columns & Sample Counts ---
Drug column detected: 'drug_name'
Unique drugs in train (12): ['bepridil', 'chlorpromazine', 'cisapride', 'diltiazem', 'dofetilide', 'mexiletine', 'ondansetron', 'quinidine', 'ranolazine', 'sotalol', 'terfenadine', 'verapamil']
Unique drugs in test  (16): ['astemizole', 'azimilide', 'clarithromycin', 'clozapine', 'disopyramide', 'domperidone', 'droperidol', 'ibutilide', 'loratadine', 'metoprolol', 'nifedipine'

## Final Summary Report

This section outputs the final summary for all checks performed.

In [5]:
print("="*60)
print("SUMMARY OF VALIDATION REPORTS")
print("="*60)
print(f"(a) Label column name:      '{label_col}'")
print(f"(b) Drug-name column name:  '{drug_col}'")
print("(c) One-line PASS/FAIL summary per dataset pair:")
for pair, status in pair_status.items():
    print(f"    - {pair}: {status}")

chantest_match = (
    pair_status["chantest"] == "PASS"
    and len(datasets["chantest_train"]) == 24000
    and len(datasets["chantest_test"]) == 32000
)
confirm_str = "CONFIRMED" if chantest_match else "FAILED TO CONFIRM"
print(f"(d) Chantest matches reference spec (24k train / 32k test, all checks PASS): {confirm_str}")

SUMMARY OF VALIDATION REPORTS
(a) Label column name:      'label'
(b) Drug-name column name:  'drug_name'
(c) One-line PASS/FAIL summary per dataset pair:
    - chantest: PASS
    - nanion: PASS
    - manual: FAIL
(d) Chantest matches reference spec (24k train / 32k test, all checks PASS): CONFIRMED


# Phase 1: Preprocessing

This phase builds clean preprocessed bundles for each dataset (`chantest`, `nanion`, `manual`) and each class scenario (`3class`, `binary`), fits standardizers on the training splits, handles NaN rows, and prepares inputs for subsequent modeling.

In [6]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

BIOMARKERS = ['qNet','dvdtmax','vmax','vrest','APD50','APD90','max_dv',
              'camax','carest','CaTD50','CaTD90']
LABEL_MAP_3CLASS = {'high':0, 'intermediate':1, 'low':2}
LABEL_MAP_BINARY = {'high':1, 'intermediate':1, 'low':0}

# Fallback: load datasets if not present in the global scope
if 'datasets' not in globals():
    import os
    print("Warning: 'datasets' dictionary not found in global scope. Loading CSVs directly from disk...")
    data_dir = "./data"
    pairs_spec = {
        "chantest": {"train": "chantest_training.csv", "test": "chantest_testing.csv"},
        "nanion": {"train": "nanion_training.csv", "test": "nanion_testing.csv"},
        "manual": {"train": "manual_training(in).csv", "test": "manual_testing(in).csv"}
    }
    datasets = {}
    for p_name, files in pairs_spec.items():
        for split in ["train", "test"]:
            f_name = files[split]
            f_path = os.path.join(data_dir, f_name)
            if os.path.exists(f_path):
                datasets[f"{p_name}_{split}"] = pd.read_csv(f_path)
            else:
                print(f"ERROR: Could not load {f_name} from {data_dir}")

prep = {}

for dataset in ['chantest', 'nanion', 'manual']:
    prep[dataset] = {}
    
    train_key = f"{dataset}_train"
    test_key = f"{dataset}_test"
    
    df_train_raw = datasets[train_key]
    df_test_raw = datasets[test_key]
    
    # Drop NaN rows in biomarker columns (separately for train and test)
    orig_train_len = len(df_train_raw)
    orig_test_len = len(df_test_raw)
    
    df_train_clean = df_train_raw.dropna(subset=BIOMARKERS)
    dropped_train = orig_train_len - len(df_train_clean)
    
    df_test_clean = df_test_raw.dropna(subset=BIOMARKERS)
    dropped_test = orig_test_len - len(df_test_clean)
    
    print(f"Dataset '{dataset}':")
    print(f"  Dropped {dropped_train} rows from Train due to NaNs (remaining: {len(df_train_clean)})")
    print(f"  Dropped {dropped_test} rows from Test due to NaNs (remaining: {len(df_test_clean)})")
    
    # Extract biomarker columns (exact order)
    X_train_raw = df_train_clean[BIOMARKERS].values
    X_test_raw = df_test_clean[BIOMARKERS].values
    
    # Fit StandardScaler on TRAIN ONLY, transform train and test
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)
    
    # Preserve test-set drug_name aligned to X_test rows
    test_drugs = df_test_clean['drug_name'].values
    
    for scenario in ['3class', 'binary']:
        if scenario == '3class':
            y_train = df_train_clean['label'].map(LABEL_MAP_3CLASS).values
            y_test = df_test_clean['label'].map(LABEL_MAP_3CLASS).values
            class_map = LABEL_MAP_3CLASS
        else:
            y_train = df_train_clean['label'].map(LABEL_MAP_BINARY).values
            y_test = df_test_clean['label'].map(LABEL_MAP_BINARY).values
            class_map = LABEL_MAP_BINARY
            
        prep[dataset][scenario] = {
            'X_train': X_train_scaled,
            'X_test': X_test_scaled,
            'y_train': y_train,
            'y_test': y_test,
            'scaler': scaler,
            'test_drugs': test_drugs,
            'class_map': class_map
        }


Dataset 'chantest':
  Dropped 0 rows from Train due to NaNs (remaining: 24000)
  Dropped 0 rows from Test due to NaNs (remaining: 32000)
Dataset 'nanion':
  Dropped 0 rows from Train due to NaNs (remaining: 24000)
  Dropped 0 rows from Test due to NaNs (remaining: 32000)
Dataset 'manual':
  Dropped 25 rows from Train due to NaNs (remaining: 23975)
  Dropped 1 rows from Test due to NaNs (remaining: 31999)


## Preprocessing Validation

We run a validation suite on the preprocessed bundles to verify: shapes, mean and standard deviation scaling (no leakage check), class distributions, aligned test drug counts/uniques, and check status.

In [7]:
import numpy as np
import pandas as pd

print("="*60)
print("PREPROCESSING VALIDATION REPORT")
print("="*60)

for dataset in ['chantest', 'nanion', 'manual']:
    for scenario in ['3class', 'binary']:
        print(f"\n--- {dataset.upper()} - {scenario.upper()} ---")
        
        bundle = prep[dataset][scenario]
        X_tr = bundle['X_train']
        X_te = bundle['X_test']
        y_tr = bundle['y_train']
        y_te = bundle['y_test']
        scl = bundle['scaler']
        tdru = bundle['test_drugs']
        
        # 1. Check Shapes
        print(f"X_train shape: {X_tr.shape}, y_train shape: {y_tr.shape}")
        print(f"X_test shape:  {X_te.shape}, y_test shape:  {y_te.shape}")
        
        shape_ok = (X_tr.shape[0] == y_tr.shape[0] and X_te.shape[0] == y_te.shape[0] and X_tr.shape[1] == len(BIOMARKERS))
        
        # 2. Check scaling
        tr_mean = X_tr.mean(axis=0)
        tr_std = X_tr.std(axis=0)
        te_mean = X_te.mean(axis=0)
        te_std = X_te.std(axis=0)
        
        print("X_train per-column mean range: ", f"[{tr_mean.min():.4f}, {tr_mean.max():.4f}]")
        print("X_train per-column std range:  ", f"[{tr_std.min():.4f}, {tr_std.max():.4f}]")
        print("X_test per-column mean range:  ", f"[{te_mean.min():.4f}, {te_mean.max():.4f}]")
        print("X_test per-column std range:   ", f"[{te_std.min():.4f}, {te_std.max():.4f}]")
        
        scaling_ok = np.allclose(tr_mean, 0, atol=1e-7) and np.allclose(tr_std, 1, atol=1e-7)
        leakage_check_ok = not (np.allclose(te_mean, 0, atol=1e-7) and np.allclose(te_std, 1, atol=1e-7))
        
        # 3. Label distribution
        unique_tr, counts_tr = np.unique(y_tr, return_counts=True)
        unique_te, counts_te = np.unique(y_te, return_counts=True)
        print("y_train distribution:", dict(zip(unique_tr, counts_tr)))
        print("y_test distribution: ", dict(zip(unique_te, counts_te)))
        
        # 4. Drug name alignment
        len_ok = len(tdru) == X_te.shape[0]
        unique_drugs = np.unique(tdru)
        print(f"test_drugs aligned length: {len(tdru)} (Match X_test: {len_ok})")
        print(f"Unique test drugs count: {len(unique_drugs)}")
        
        drug_counts = pd.Series(tdru).value_counts()
        print(f"Per-drug sample counts range: Min={drug_counts.min()}, Max={drug_counts.max()}")
        
        # A PASS/FAIL line per dataset*scenario
        if dataset == 'manual':
            expected_tr_rows = 24000 - 25
            expected_te_rows = 32000 - 1
            expected_unique_drugs = 16
            expected_min_drug = 1999
        else:
            expected_tr_rows = 24000
            expected_te_rows = 32000
            expected_unique_drugs = 16
            expected_min_drug = 2000
            
        pass_fail = (
            shape_ok and scaling_ok and leakage_check_ok and len_ok 
            and X_tr.shape[0] == expected_tr_rows 
            and X_te.shape[0] == expected_te_rows
            and len(unique_drugs) == expected_unique_drugs
            and drug_counts.min() == expected_min_drug
        )
        status = "PASS" if pass_fail else "FAIL"
        print(f"Check Status: {dataset} - {scenario} -> {status}")


PREPROCESSING VALIDATION REPORT

--- CHANTEST - 3CLASS ---
X_train shape: (24000, 11), y_train shape: (24000,)
X_test shape:  (32000, 11), y_test shape:  (32000,)
X_train per-column mean range:  [-0.0000, 0.0000]
X_train per-column std range:   [1.0000, 1.0000]
X_test per-column mean range:   [-0.4291, 0.5450]
X_test per-column std range:    [0.2002, 2.3476]
y_train distribution: {np.int64(0): np.int64(8000), np.int64(1): np.int64(8000), np.int64(2): np.int64(8000)}
y_test distribution:  {np.int64(0): np.int64(8000), np.int64(1): np.int64(14000), np.int64(2): np.int64(10000)}
test_drugs aligned length: 32000 (Match X_test: True)
Unique test drugs count: 16
Per-drug sample counts range: Min=2000, Max=2000
Check Status: chantest - 3class -> PASS

--- CHANTEST - BINARY ---
X_train shape: (24000, 11), y_train shape: (24000,)
X_test shape:  (32000, 11), y_test shape:  (32000,)
X_train per-column mean range:  [-0.0000, 0.0000]
X_train per-column std range:   [1.0000, 1.0000]
X_test per-colum

# Installation and Verification of Deep-Learning Dependencies

We install the requested packages `tensorflow` and `scikeras` via pip, and print their versions to verify successful import.

In [8]:
import subprocess
import sys

print("Installing tensorflow and scikeras...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow", "scikeras"])
    print("Installation command executed successfully.\n")
except Exception as e:
    print(f"Installation failed: {e}\n")

try:
    import tensorflow as tf
    print(f"tensorflow version: {tf.__version__}")
except Exception as e:
    print(f"Failed to import tensorflow: {e}")

try:
    import scikeras
    print(f"scikeras version: {scikeras.__version__}")
except Exception as e:
    print(f"Failed to import scikeras: {e}")


Installing tensorflow and scikeras...
Installation command executed successfully.

tensorflow version: 2.21.0
scikeras version: 0.13.0


# Phase 2: Base Learners & Tuning

In this phase, we tune the hyper-parameters for four machine learning learners (`RF`, `XGB`, `LR`, `SVM`) and three deep learning neural networks (`ANN1`, `ANN2`, `ANN3`) using `GridSearchCV` with 3-fold stratified cross-validation on the training set. The tuned models are then persisted to `./artifacts/base_models.joblib`.

In [9]:
import time
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set seeds for reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Optimize thread settings for CPU execution
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

# Keras sequential build function using layers.Input to prevent warnings
def build_ann(n_hidden_layers=1, nodes=32, lr=0.001, num_classes=3):
    model = keras.Sequential()
    model.add(layers.Input(shape=(11,)))
    for _ in range(n_hidden_layers):
        model.add(layers.Dense(nodes, activation='relu'))
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cv_splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

base_models = {}
datasets_list = ['chantest', 'nanion', 'manual']
scenarios_list = ['3class', 'binary']

artifacts_dir = "./artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

print("=== Phase 2: Base Learners Hyperparameter Tuning ===\n")


=== Phase 2: Base Learners Hyperparameter Tuning ===



In [10]:
for dataset in datasets_list:
    base_models[dataset] = {}
    for scenario in scenarios_list:
        t_start = time.time()
        print(f"Tuning base learners for {dataset.upper()} ({scenario.upper()})...")
        
        bundle = prep[dataset][scenario]
        X_train = bundle['X_train']
        y_train = bundle['y_train']
        
        num_classes = 3 if scenario == '3class' else 2
        xgb_obj = 'multi:softprob' if scenario == '3class' else 'binary:logistic'
        
        # 1. Random Forest
        rf = RandomForestClassifier(random_state=SEED)
        rf_grid = {
            'criterion': ['gini', 'entropy'],
            'n_estimators': [200, 400],
            'max_depth': [None, 20]
        }
        
        # 2. XGBoost
        xgb_model = xgb.XGBClassifier(
            random_state=SEED, 
            tree_method='hist', 
            objective=xgb_obj,
            num_class=num_classes if scenario == '3class' else None
        )
        xgb_grid = {
            'learning_rate': [0.001, 0.1],
            'max_depth': [6, 10],
            'n_estimators': [300, 500]
        }
        
        # 3. Logistic Regression
        lr_model = LogisticRegression(max_iter=2000, random_state=SEED)
        lr_grid = {
            'C': [0.01, 1.0],
            'penalty': ['l2'],
            'solver': ['lbfgs']
        }
        
        # 4. SVM
        svm_model = SVC(probability=True, random_state=SEED)
        svm_grid = {
            'C': [1.0, 10.0],
            'gamma': ['scale', 'auto']
        }
        
        # 5. ANN1
        ann1 = KerasClassifier(
            model=build_ann,
            model__n_hidden_layers=1,
            model__num_classes=num_classes,
            epochs=30,
            batch_size=64,
            verbose=0,
            random_state=SEED
        )
        ann1_grid = {
            'model__nodes': [32, 64],
            'model__lr': [0.0001, 0.01]
        }
        
        # 6. ANN2
        ann2 = KerasClassifier(
            model=build_ann,
            model__n_hidden_layers=2,
            model__num_classes=num_classes,
            epochs=30,
            batch_size=64,
            verbose=0,
            random_state=SEED
        )
        ann2_grid = {
            'model__nodes': [64, 128],
            'model__lr': [0.01]
        }
        
        # 7. ANN3
        ann3 = KerasClassifier(
            model=build_ann,
            model__n_hidden_layers=3,
            model__num_classes=num_classes,
            epochs=30,
            batch_size=64,
            verbose=0,
            random_state=SEED
        )
        ann3_grid = {
            'model__nodes': [128],
            'model__lr': [0.01]
        }
        
        learners = {
            'RF': (rf, rf_grid, -1),
            'XGB': (xgb_model, xgb_grid, -1),
            'LR': (lr_model, lr_grid, -1),
            'SVM': (svm_model, svm_grid, -1),
            'ANN1': (ann1, ann1_grid, 1),
            'ANN2': (ann2, ann2_grid, 1),
            'ANN3': (ann3, ann3_grid, 1)
        }
        
        base_models[dataset][scenario] = {}
        
        for name, (model, grid, nj) in learners.items():
            t0 = time.time()
            grid_search = GridSearchCV(
                estimator=model, 
                param_grid=grid, 
                cv=cv_splitter, 
                scoring='f1_macro', 
                n_jobs=nj, 
                refit=True
            )
            grid_search.fit(X_train, y_train)
            t1 = time.time()
            
            if name.startswith('ANN'):
                keras_dir = os.path.join(artifacts_dir, 'keras')
                os.makedirs(keras_dir, exist_ok=True)
                keras_path = os.path.join(keras_dir, f'{dataset}_{scenario}_{name}.keras')
                grid_search.best_estimator_.model_.save(keras_path)
                base_models[dataset][scenario][name] = {
                    'estimator': keras_path,
                    'best_params': grid_search.best_params_,
                    'cv_f1_macro': float(grid_search.best_score_)
                }
            else:
                base_models[dataset][scenario][name] = {
                    'estimator': grid_search.best_estimator_,
                    'best_params': grid_search.best_params_,
                    'cv_f1_macro': float(grid_search.best_score_)
                }
            print(f"  {name} tuned in {t1-t0:.2f} seconds.")
            
        elapsed = time.time() - t_start
        base_models[dataset][scenario]['wall_clock_time'] = elapsed
        print(f"Finished tuning {dataset.upper()} ({scenario.upper()}) in {elapsed:.2f} seconds.\n")


Tuning base learners for CHANTEST (3CLASS)...
  RF tuned in 55.22 seconds.
  XGB tuned in 19.60 seconds.
  LR tuned in 0.44 seconds.
  SVM tuned in 6.45 seconds.
  ANN1 tuned in 162.39 seconds.
  ANN2 tuned in 95.90 seconds.
  ANN3 tuned in 78.24 seconds.
Finished tuning CHANTEST (3CLASS) in 418.37 seconds.

Tuning base learners for CHANTEST (BINARY)...
  RF tuned in 64.51 seconds.
  XGB tuned in 9.63 seconds.
  LR tuned in 0.25 seconds.
  SVM tuned in 4.96 seconds.
  ANN1 tuned in 201.37 seconds.
  ANN2 tuned in 145.77 seconds.
  ANN3 tuned in 80.15 seconds.
Finished tuning CHANTEST (BINARY) in 506.80 seconds.

Tuning base learners for NANION (3CLASS)...
  RF tuned in 82.18 seconds.
  XGB tuned in 36.39 seconds.
  LR tuned in 0.62 seconds.
  SVM tuned in 9.51 seconds.
  ANN1 tuned in 219.22 seconds.
  ANN2 tuned in 142.73 seconds.
  ANN3 tuned in 88.08 seconds.
Finished tuning NANION (3CLASS) in 578.96 seconds.

Tuning base learners for NANION (BINARY)...
  RF tuned in 64.79 seconds.


C:\Users\Mitch\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Mitch\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Mitch

## Tuning Summary & Model Persistence

We print a clean summary table of the tuning results and persist the entire `base_models` dictionary to disk.

In [11]:
print("="*80)
print("TUNING SUMMARY REPORT")
print("="*80)

for dataset in datasets_list:
    for scenario in scenarios_list:
        print(f"\nDataset: {dataset.upper()} | Scenario: {scenario.upper()}")
        print("-" * 80)
        print(f"{'Learner':<8} | {'Best CV F1 Macro':<18} | {'Best Parameters'}")
        print("-" * 80)
        for name in ['RF', 'XGB', 'LR', 'SVM', 'ANN1', 'ANN2', 'ANN3']:
            info = base_models[dataset][scenario][name]
            score = info['cv_f1_macro']
            params = info['best_params']
            print(f"{name:<8} | {score:<18.4f} | {params}")
        
        elapsed = base_models[dataset][scenario]['wall_clock_time']
        print(f"Total Wall-clock Time: {elapsed:.2f} seconds")
        print("=" * 80)
        
save_path = os.path.join(artifacts_dir, "base_models.joblib")
print(f"\nSaving base_models to {save_path}...")
joblib.dump(base_models, save_path)
print("Save successful.")


TUNING SUMMARY REPORT

Dataset: CHANTEST | Scenario: 3CLASS
--------------------------------------------------------------------------------
Learner  | Best CV F1 Macro   | Best Parameters
--------------------------------------------------------------------------------
RF       | 0.9996             | {'criterion': 'entropy', 'max_depth': None, 'n_estimators': 200}
XGB      | 0.9994             | {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300}
LR       | 0.9972             | {'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'}
SVM      | 0.9993             | {'C': 10.0, 'gamma': 'scale'}
ANN1     | 0.9988             | {'model__lr': 0.01, 'model__nodes': 64}
ANN2     | 0.9993             | {'model__lr': 0.01, 'model__nodes': 128}
ANN3     | 0.9989             | {'model__lr': 0.01, 'model__nodes': 128}
Total Wall-clock Time: 418.37 seconds

Dataset: CHANTEST | Scenario: BINARY
--------------------------------------------------------------------------------
Learner  | Best CV F1 Mac

# Phase 2.5: CV Leakage Diagnostic

This section performs a diagnostic check to determine whether the near-1.0 cross-validation scores observed in Phase 2 are caused by drug-level leakage. Since each drug contains 2,000 highly correlated simulated samples, a standard row-level split (StratifiedKFold) scatters samples of the same drug across both training and validation folds. This allows classifiers to memorize drug-specific signatures and yields overly optimistic validation scores. We compare the row-level split against a drug-level split (GroupKFold) where whole drugs are held out from the validation set.

In [10]:
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# 1. Recover groups_train (per-row drug_name) from the original chantest training DataFrame
# aligned to the same rows used in prep (i.e. after same NaN handling)
df_train_raw = datasets['chantest_train']
BIOMARKERS = ['qNet', 'dvdtmax', 'vmax', 'vrest', 'APD50', 'APD90', 'max_dv', 'camax', 'carest', 'CaTD50', 'CaTD90']
df_train_clean = df_train_raw.dropna(subset=BIOMARKERS)
groups_train = df_train_clean['drug_name'].values

# 2. Pull X_train, y_train from prep['chantest']['3class']
X_train = prep['chantest']['3class']['X_train']
y_train = prep['chantest']['3class']['y_train']

# 3. Instantiate fresh RandomForestClassifier
rf_diagnostic = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

# 4. Scheme A: Row-level cross-validation
cv_a = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores_a = cross_val_score(rf_diagnostic, X_train, y_train, cv=cv_a, scoring='f1_macro')

# 5. Scheme B: Drug-level cross-validation
cv_b = GroupKFold(n_splits=3)
scores_b = cross_val_score(rf_diagnostic, X_train, y_train, cv=cv_b, groups=groups_train, scoring='f1_macro')

# 6. Count unique drugs in each fold's validation set under Scheme B
unique_drugs_per_fold = []
for _, val_idx in cv_b.split(X_train, y_train, groups=groups_train):
    unique_drugs_per_fold.append(len(np.unique(groups_train[val_idx])))

# 7. Print results
print(f"Scheme A (Row-level StratifiedKFold) - Mean F1 Macro: {scores_a.mean():.6f}")
print(f"Scheme A - Per-fold F1 Macro: {scores_a}")
print(f"Scheme B (Drug-level GroupKFold) - Mean F1 Macro: {scores_b.mean():.6f}")
print(f"Scheme B - Per-fold F1 Macro: {scores_b}")
print(f"Unique drugs in each validation fold under Scheme B: {unique_drugs_per_fold}")

# 8. One-line verdict
if scores_a.mean() > 0.95 and scores_b.mean() < 0.80:
    print("Verdict: Drug-level leakage is confirmed (row-level CV is near 1.0, while drug-level CV is materially lower).")
else:
    print("Verdict: Drug-level leakage is not confirmed by this threshold.")


Scheme A (Row-level StratifiedKFold) - Mean F1 Macro: 0.999542
Scheme A - Per-fold F1 Macro: [0.99987502 0.99924998 0.99950006]
Scheme B (Drug-level GroupKFold) - Mean F1 Macro: 0.460310
Scheme B - Per-fold F1 Macro: [0.28692507 0.63040269 0.46360319]
Unique drugs in each validation fold under Scheme B: [4, 4, 4]
Verdict: Drug-level leakage is confirmed (row-level CV is near 1.0, while drug-level CV is materially lower).


# Phase 2 Re-tune: Grouped CV

Based on the results from Phase 2.5, the original row-level CV scores were severely inflated by drug-level leakage. We now re-tune all 7 base learners using a leakage-free 4-fold `StratifiedGroupKFold` cross-validation scheme grouped by the `drug_name` column. This prevents any drug's samples from appearing in both the training and validation sets, ensuring our cross-validation performance correctly reflects generalization to unseen drugs. Outputs are saved incrementally to `./artifacts/base_models_grouped.joblib` and `./artifacts/keras_grouped/`.

In [11]:
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time
import os
import joblib
import numpy as np
import pandas as pd

# Set seeds for reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Optimize thread settings for CPU execution
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

# 1. Recover train groups
train_groups = {}
BIOMARKERS = ['qNet', 'dvdtmax', 'vmax', 'vrest', 'APD50', 'APD90', 'max_dv', 'camax', 'carest', 'CaTD50', 'CaTD90']
for dataset in ['chantest', 'nanion', 'manual']:
    df_raw = datasets[f'{dataset}_train']
    df_clean = df_raw.dropna(subset=BIOMARKERS)
    train_groups[dataset] = df_clean['drug_name'].values

# Load original base_models for the comparison table later
base_models = joblib.load('./artifacts/base_models.joblib')

# Keras sequential build function
def build_ann(n_hidden_layers=1, nodes=32, lr=0.001, num_classes=3):
    model = keras.Sequential()
    model.add(layers.Input(shape=(11,)))
    for _ in range(n_hidden_layers):
        model.add(layers.Dense(nodes, activation='relu'))
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

base_models_grouped = {}
datasets_list = ['chantest', 'nanion', 'manual']
scenarios_list = ['3class', 'binary']
artifacts_dir = './artifacts'
keras_grouped_dir = os.path.join(artifacts_dir, 'keras_grouped')
os.makedirs(keras_grouped_dir, exist_ok=True)

cv_splitter_grouped = StratifiedGroupKFold(n_splits=4)

total_tuning_start = time.time()
print('=== Phase 2 Re-tune: Leakage-free Grouped CV Hyperparameter Tuning ===\n')

for dataset in datasets_list:
    base_models_grouped[dataset] = {}
    for scenario in scenarios_list:
        t_start = time.time()
        print(f'Tuning base learners for {dataset.upper()} ({scenario.upper()}) using StratifiedGroupKFold...')
        
        bundle = prep[dataset][scenario]
        X_train = bundle['X_train']
        y_train = bundle['y_train']
        groups = train_groups[dataset]
        
        num_classes = 3 if scenario == '3class' else 2
        xgb_obj = 'multi:softprob' if scenario == '3class' else 'binary:logistic'
        
        # 1. Random Forest
        rf = RandomForestClassifier(random_state=SEED)
        rf_grid = {
            'criterion': ['gini', 'entropy'],
            'n_estimators': [200, 400],
            'max_depth': [None, 20]
        }
        
        # 2. XGBoost
        xgb_model = xgb.XGBClassifier(
            random_state=SEED, 
            tree_method='hist', 
            objective=xgb_obj,
            num_class=num_classes if scenario == '3class' else None
        )
        xgb_grid = {
            'learning_rate': [0.001, 0.1],
            'max_depth': [6, 10],
            'n_estimators': [300, 500]
        }
        
        # 3. Logistic Regression
        lr_model = LogisticRegression(max_iter=2000, random_state=SEED)
        lr_grid = {
            'C': [0.01, 1.0],
            'penalty': ['l2'],
            'solver': ['lbfgs']
        }
        
        # 4. SVM
        svm_model = SVC(probability=True, random_state=SEED)
        svm_grid = {
            'C': [1.0, 10.0],
            'gamma': ['scale', 'auto']
        }
        
        # 5. ANN1
        ann1 = KerasClassifier(
            model=build_ann,
            model__n_hidden_layers=1,
            model__num_classes=num_classes,
            epochs=30,
            batch_size=64,
            verbose=0,
            random_state=SEED
        )
        ann1_grid = {
            'model__nodes': [32, 64],
            'model__lr': [0.0001, 0.01]
        }
        
        # 6. ANN2
        ann2 = KerasClassifier(
            model=build_ann,
            model__n_hidden_layers=2,
            model__num_classes=num_classes,
            epochs=30,
            batch_size=64,
            verbose=0,
            random_state=SEED
        )
        ann2_grid = {
            'model__nodes': [64, 128],
            'model__lr': [0.01]
        }
        
        # 7. ANN3
        ann3 = KerasClassifier(
            model=build_ann,
            model__n_hidden_layers=3,
            model__num_classes=num_classes,
            epochs=30,
            batch_size=64,
            verbose=0,
            random_state=SEED
        )
        ann3_grid = {
            'model__nodes': [128],
            'model__lr': [0.01]
        }
        
        learners = {
            'RF': (rf, rf_grid, -1),
            'XGB': (xgb_model, xgb_grid, -1),
            'LR': (lr_model, lr_grid, -1),
            'SVM': (svm_model, svm_grid, -1),
            'ANN1': (ann1, ann1_grid, 1),
            'ANN2': (ann2, ann2_grid, 1),
            'ANN3': (ann3, ann3_grid, 1)
        }
        
        base_models_grouped[dataset][scenario] = {}
        
        for name, (model, grid, nj) in learners.items():
            t0 = time.time()
            grid_search = GridSearchCV(
                estimator=model, 
                param_grid=grid, 
                cv=cv_splitter_grouped, 
                scoring='f1_macro', 
                n_jobs=nj, 
                refit=True
            )
            grid_search.fit(X_train, y_train, groups=groups)
            t1 = time.time()
            
            if name.startswith('ANN'):
                keras_path = os.path.join(keras_grouped_dir, f'{dataset}_{scenario}_{name}.keras')
                grid_search.best_estimator_.model_.save(keras_path)
                base_models_grouped[dataset][scenario][name] = {
                    'estimator': keras_path,
                    'best_params': grid_search.best_params_,
                    'cv_f1_macro': float(grid_search.best_score_)
                }
            else:
                base_models_grouped[dataset][scenario][name] = {
                    'estimator': grid_search.best_estimator_,
                    'best_params': grid_search.best_params_,
                    'cv_f1_macro': float(grid_search.best_score_)
                }
            print(f'  {name} tuned in {t1-t0:.2f} seconds.')
            
        elapsed = time.time() - t_start
        base_models_grouped[dataset][scenario]['wall_clock_time'] = elapsed
        print(f'Finished tuning {dataset.upper()} ({scenario.upper()}) in {elapsed:.2f} seconds.')
        
        # Save progress immediately after finishing each dataset x scenario
        joblib.dump(base_models_grouped, os.path.join(artifacts_dir, 'base_models_grouped.joblib'))
        print(f'checkpoint saved: {dataset} x {scenario}')

total_tuning_time = time.time() - total_tuning_start
print(f'Total hyperparameter tuning complete. Time: {total_tuning_time/60:.2f} minutes\n')

# Print tables
print('='*80)
print('TUNING SUMMARY REPORT: GROUPED CV')
print('='*80)
for dataset in datasets_list:
    for scenario in scenarios_list:
        print(f'\nDataset: {dataset.upper()} | Scenario: {scenario.upper()}')
        print('-' * 80)
        print(f"{'Learner':<8} | {'Grouped CV F1 Macro':<20} | {'Best Parameters'}")
        print('-' * 80)
        for name in ['RF', 'XGB', 'LR', 'SVM', 'ANN1', 'ANN2', 'ANN3']:
            info = base_models_grouped[dataset][scenario][name]
            score = info['cv_f1_macro']
            params = info['best_params']
            print(f'{name:<8} | {score:<20.4f} | {params}')
        elapsed = base_models_grouped[dataset][scenario]['wall_clock_time']
        print(f'Total Wall-clock Time: {elapsed:.2f} seconds')
        print('=' * 80)

# Print side-by-side comparison
print('\n' + '='*80)
print('SIDE-BY-SIDE COMPARISON: OLD LEAKY CV vs NEW GROUPED CV')
print('='*80)
for dataset in datasets_list:
    for scenario in scenarios_list:
        print(f'\nDataset: {dataset.upper()} | Scenario: {scenario.upper()}')
        print(f"{'Learner':<8} | {'Old Leaky CV F1':<18} | {'New Grouped CV F1'}")
        print('-' * 50)
        for name in ['RF', 'XGB', 'LR', 'SVM', 'ANN1', 'ANN2', 'ANN3']:
            old_score = base_models[dataset][scenario][name]['cv_f1_macro']
            new_score = base_models_grouped[dataset][scenario][name]['cv_f1_macro']
            print(f'{name:<8} | {old_score:<18.4f} | {new_score:<18.4f}')
        print('-' * 50)


=== Phase 2 Re-tune: Leakage-free Grouped CV Hyperparameter Tuning ===

Tuning base learners for CHANTEST (3CLASS) using StratifiedGroupKFold...
  RF tuned in 106.46 seconds.
  XGB tuned in 27.40 seconds.
  LR tuned in 0.58 seconds.
  SVM tuned in 8.04 seconds.
  ANN1 tuned in 333.45 seconds.
  ANN2 tuned in 227.30 seconds.
  ANN3 tuned in 175.91 seconds.
Finished tuning CHANTEST (3CLASS) in 879.31 seconds.
checkpoint saved: chantest x 3class
Tuning base learners for CHANTEST (BINARY) using StratifiedGroupKFold...
  RF tuned in 73.61 seconds.
  XGB tuned in 7.79 seconds.
  LR tuned in 0.26 seconds.
  SVM tuned in 5.21 seconds.
  ANN1 tuned in 250.28 seconds.
  ANN2 tuned in 229.77 seconds.
  ANN3 tuned in 180.52 seconds.
Finished tuning CHANTEST (BINARY) in 747.54 seconds.
checkpoint saved: chantest x binary
Tuning base learners for NANION (3CLASS) using StratifiedGroupKFold...
  RF tuned in 62.83 seconds.
  XGB tuned in 26.06 seconds.
  LR tuned in 0.49 seconds.
  SVM tuned in 12.57 s

C:\Users\Mitch\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
1 fits failed out of a total of 32.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Mitch\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Mitch\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packag

# Phase 3: Ensemble Sweep

In this phase, we perform a combinatorial ensemble sweep across 17 unique model combinations (11 pure-ML, 4 pure-DL, and 2 mixed reference baselines) and 6 ensemble configurations (2 Voting, 4 Stacking). Stacking meta-learners are tuned using 4-fold `StratifiedGroupKFold` grouped by `drug_name` to prevent leakage. For each combination and configuration, we train on the training split and evaluate on the real test split (Protocol A evaluation). Metrics are saved incrementally to `./artifacts/phase3_protocolA_results.csv`.

In [12]:
import sklearn
sklearn.set_config(enable_metadata_routing=True)

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, recall_score, accuracy_score
import tensorflow as tf
import numpy as np
import pandas as pd
import time
import os
import joblib
import itertools

# Set seeds for reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Optimize thread settings for CPU execution
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

class FittedHolder:
    def __init__(self, estimator):
        self.estimator = estimator

class PreFittedEstimatorWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, holder=None):
        self.holder = holder
        if holder is not None:
            self.classes_ = holder.estimator.classes_
            
    def fit(self, X, y, sample_weight=None, groups=None):
        if self.holder is not None:
            self.classes_ = self.holder.estimator.classes_
        return self
        
    def predict(self, X):
        return self.holder.estimator.predict(X)
        
    def predict_proba(self, X):
        return self.holder.estimator.predict_proba(X)

class KerasSklearnWrapper:
    def __init__(self, keras_model, num_classes):
        self.model = keras_model
        self.classes_ = np.arange(num_classes)
        
    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)
        
    def predict_proba(self, X):
        return self.model.predict(X, verbose=0)

class FinalEstimatorWrapper(ClassifierMixin, BaseEstimator):
    _estimator_type = 'classifier'
    def __init__(self, estimator=None):
        self.estimator = estimator
        if estimator is not None and hasattr(estimator, 'classes_'):
            self.classes_ = estimator.classes_
            
    def fit(self, X, y, sample_weight=None, groups=None, **kwargs):
        fit_params = {}
        if sample_weight is not None:
            fit_params['sample_weight'] = sample_weight
        self.estimator.fit(X, y, **fit_params)
        self.classes_ = self.estimator.classes_
        return self
        
    def predict(self, X):
        return self.estimator.predict(X)
        
    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

# 1. Recover groups_train (per-row drug_name) from the original training DataFrame
train_groups = {}
BIOMARKERS = ['qNet', 'dvdtmax', 'vmax', 'vrest', 'APD50', 'APD90', 'max_dv', 'camax', 'carest', 'CaTD50', 'CaTD90']
for ds in ['chantest', 'nanion', 'manual']:
    df_raw = datasets[f'{ds}_train']
    df_clean = df_raw.dropna(subset=BIOMARKERS)
    train_groups[ds] = df_clean['drug_name'].values

base_models_grouped = joblib.load('./artifacts/base_models_grouped.joblib')
keras_grouped_dir = './artifacts/keras_grouped'

def load_base_estimators(dataset, scenario):
    num_classes = 3 if scenario == '3class' else 2
    
    # ML Models
    rf_est = base_models_grouped[dataset][scenario]['RF']['estimator']
    xgb_est = base_models_grouped[dataset][scenario]['XGB']['estimator']
    lr_est = base_models_grouped[dataset][scenario]['LR']['estimator']
    svm_est = base_models_grouped[dataset][scenario]['SVM']['estimator']
    
    # Wrappers for ML
    w_rf = PreFittedEstimatorWrapper(FittedHolder(rf_est))
    w_xgb = PreFittedEstimatorWrapper(FittedHolder(xgb_est))
    w_lr = PreFittedEstimatorWrapper(FittedHolder(lr_est))
    w_svm = PreFittedEstimatorWrapper(FittedHolder(svm_est))
    
    # DL Models
    ann1_path = os.path.join(keras_grouped_dir, f'{dataset}_{scenario}_ANN1.keras')
    ann2_path = os.path.join(keras_grouped_dir, f'{dataset}_{scenario}_ANN2.keras')
    ann3_path = os.path.join(keras_grouped_dir, f'{dataset}_{scenario}_ANN3.keras')
    
    ann1_model = tf.keras.models.load_model(ann1_path)
    ann2_model = tf.keras.models.load_model(ann2_path)
    ann3_model = tf.keras.models.load_model(ann3_path)
    
    w_ann1 = PreFittedEstimatorWrapper(FittedHolder(KerasSklearnWrapper(ann1_model, num_classes)))
    w_ann2 = PreFittedEstimatorWrapper(FittedHolder(KerasSklearnWrapper(ann2_model, num_classes)))
    w_ann3 = PreFittedEstimatorWrapper(FittedHolder(KerasSklearnWrapper(ann3_model, num_classes)))
    
    # Set metadata fit request
    for w in [w_rf, w_xgb, w_lr, w_svm, w_ann1, w_ann2, w_ann3]:
        w.set_fit_request(groups=True)
        
    return {
        'RF': w_rf,
        'XGB': w_xgb,
        'LR': w_lr,
        'SVM': w_svm,
        'ANN1': w_ann1,
        'ANN2': w_ann2,
        'ANN3': w_ann3
    }

def make_ensemble(combo_elements, config_type, base_estimators_dict, train_groups_dataset, SEED):
    estimators = [(name, base_estimators_dict[name]) for name in combo_elements]
    
    if config_type == 'hard':
        return VotingClassifier(estimators=estimators, voting='hard')
    elif config_type == 'soft':
        return VotingClassifier(estimators=estimators, voting='soft')
    elif config_type.startswith('meta='):
        meta_name = config_type.split('=')[1]
        
        if meta_name == 'LR':
            final_est = LogisticRegression(max_iter=2000, random_state=SEED)
        elif meta_name == 'SVC':
            final_est = SVC(probability=True, random_state=SEED)
        elif meta_name == 'RF':
            final_est = RandomForestClassifier(random_state=SEED)
        elif meta_name == 'XGB':
            final_est = xgb.XGBClassifier(random_state=SEED, tree_method='hist')
            
        final_est_wrapped = FinalEstimatorWrapper(final_est)
        
        cv = StratifiedGroupKFold(n_splits=4)
        cv.set_split_request(groups=True)
        
        return StackingClassifier(
            estimators=estimators,
            final_estimator=final_est_wrapped,
            cv=cv,
            stack_method='predict_proba',
            passthrough=False
        )

# Define combinations
ml_elements = ['RF', 'XGB', 'LR', 'SVM']
dl_elements = ['ANN1', 'ANN2', 'ANN3']

combos = []

# 1. Pure ML subsets (size >= 2)
for r in range(2, 5):
    for c in itertools.combinations(ml_elements, r):
        combos.append({
            'name': '+'.join(c),
            'elements': list(c),
            'paradigm': 'ML',
            'is_reference': False
        })
        
# 2. Pure DL subsets (size >= 2)
for r in range(2, 4):
    for c in itertools.combinations(dl_elements, r):
        combos.append({
            'name': '+'.join(c),
            'elements': list(c),
            'paradigm': 'DL',
            'is_reference': (c == ('ANN1', 'ANN2', 'ANN3'))
        })

# 3. Add mixed-all-7 reference
combos.append({
    'name': 'mixed-all-7',
    'elements': ml_elements + dl_elements,
    'paradigm': 'mixed',
    'is_reference': True
})

# 4. Add paper-faithful 3-ANN reference
combos.append({
    'name': 'paper-faithful 3-ANN',
    'elements': dl_elements,
    'paradigm': 'DL',
    'is_reference': True
})

results_list = []
failures = []
datasets_list = ['chantest', 'nanion', 'manual']
scenarios_list = ['3class', 'binary']
configs = ['hard', 'soft', 'meta=LR', 'meta=SVC', 'meta=RF', 'meta=XGB']

total_sweep_start = time.time()

for dataset in datasets_list:
    for scenario in scenarios_list:
        print(f'Running ensemble sweep for {dataset} x {scenario}...')
        
        # Load prep arrays
        bundle = prep[dataset][scenario]
        X_train = bundle['X_train']
        y_train = bundle['y_train']
        X_test = bundle['X_test']
        y_test = bundle['y_test']
        groups = train_groups[dataset]
        
        # Load base estimators
        try:
            base_estimators = load_base_estimators(dataset, scenario)
        except Exception as e:
            print(f'Error loading base estimators for {dataset} {scenario}: {e}')
            failures.append({
                'dataset': dataset,
                'scenario': scenario,
                'combo': 'ALL',
                'config': 'LOAD',
                'error': str(e)
            })
            continue
            
        for combo in combos:
            for config in configs:
                ensemble_family = 'voting' if config in ['hard', 'soft'] else 'stacking'
                
                try:
                    # Create ensemble
                    model = make_ensemble(combo['elements'], config, base_estimators, groups, SEED)
                    
                    # Fit ensemble
                    if ensemble_family == 'voting':
                        model.fit(X_train, y_train)
                    else:
                        model.fit(X_train, y_train, groups=groups)
                        
                    # Predict and evaluate
                    y_pred = model.predict(X_test)
                    
                    test_macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
                    f1_per_class = f1_score(y_test, y_pred, average=None, zero_division=0)
                    recall_per_class = recall_score(y_test, y_pred, average=None, zero_division=0)
                    accuracy = accuracy_score(y_test, y_pred)
                    
                    # Map class specific metrics
                    if scenario == '3class':
                        f1_high = f1_per_class[0]
                        f1_intermediate = f1_per_class[1]
                        f1_low = f1_per_class[2]
                        recall_high = recall_per_class[0]
                        recall_intermediate = recall_per_class[1]
                        recall_low = recall_per_class[2]
                    else:
                        f1_high = f1_per_class[1]
                        f1_intermediate = np.nan
                        f1_low = f1_per_class[0]
                        recall_high = recall_per_class[1]
                        recall_intermediate = np.nan
                        recall_low = recall_per_class[0]
                        
                    results_list.append({
                        'dataset': dataset,
                        'scenario': scenario,
                        'ensemble_family': ensemble_family,
                        'combo': combo['name'],
                        'detail': config,
                        'n_base_learners': len(combo['elements']),
                        'paradigm': combo['paradigm'],
                        'test_macro_f1': test_macro_f1,
                        'f1_high': f1_high,
                        'f1_intermediate': f1_intermediate,
                        'f1_low': f1_low,
                        'recall_high': recall_high,
                        'recall_intermediate': recall_intermediate,
                        'recall_low': recall_low,
                        'accuracy': accuracy
                    })
                    
                except Exception as e:
                    print(f'Failure on {dataset} {scenario} | Combo: {combo["name"]} | Config: {config} | Error: {e}')
                    failures.append({
                        'dataset': dataset,
                        'scenario': scenario,
                        'combo': combo['name'],
                        'config': config,
                        'error': str(e)
                    })
                    
        # Checkpoint save
        results_A = pd.DataFrame(results_list)
        results_A.to_csv('./artifacts/phase3_protocolA_results.csv', index=False)
        print(f'checkpoint: {dataset} {scenario} done')

total_sweep_time = time.time() - total_sweep_start
print(f'\nEnsemble Sweep Complete! Total Runtime: {total_sweep_time/60:.2f} minutes')

# Report failures
if failures:
    print(f'\nFailures encountered ({len(failures)}):')
    for f in failures:
        print(f"  - Dataset: {f['dataset']}, Scenario: {f['scenario']}, Combo: {f['combo']}, Config: {f['config']}, Error: {f['error']}")
else:
    print('\nNo failures encountered during the ensemble sweep.')

# Focused Output
df_results = pd.DataFrame(results_list)
df_chantest_3class = df_results[(df_results['dataset'] == 'chantest') & (df_results['scenario'] == '3class')]
df_sorted = df_chantest_3class.sort_values(by='test_macro_f1', ascending=False)

print('\n' + '='*80)
print('TOP 10 CONFIGS FOR CHANTEST/3CLASS:')
print('='*80)
print(df_sorted.head(10)[['combo', 'ensemble_family', 'detail', 'test_macro_f1', 'accuracy']].to_string(index=False))

print('\n' + '='*80)
print('BEST STACKING & VOTING CONFIGS FOR CHANTEST/3CLASS:')
print('='*80)
best_stacking = df_sorted[df_sorted['ensemble_family'] == 'stacking'].iloc[0]
best_voting = df_sorted[df_sorted['ensemble_family'] == 'voting'].iloc[0]

print('BEST STACKING:')
print(best_stacking.to_string())
print('\nBEST VOTING:')
print(best_voting.to_string())


Running ensemble sweep for chantest x 3class...
checkpoint: chantest 3class done
Running ensemble sweep for chantest x binary...
checkpoint: chantest binary done
Running ensemble sweep for nanion x 3class...
checkpoint: nanion 3class done
Running ensemble sweep for nanion x binary...
checkpoint: nanion binary done
Running ensemble sweep for manual x 3class...
checkpoint: manual 3class done
Running ensemble sweep for manual x binary...
checkpoint: manual binary done

Ensemble Sweep Complete! Total Runtime: 51.20 minutes

No failures encountered during the ensemble sweep.

TOP 10 CONFIGS FOR CHANTEST/3CLASS:
        combo ensemble_family   detail  test_macro_f1  accuracy
       RF+XGB        stacking  meta=RF       0.828004  0.824187
    RF+XGB+LR        stacking  meta=RF       0.827086  0.823406
   RF+XGB+SVM        stacking  meta=RF       0.826735  0.823063
        RF+LR        stacking  meta=RF       0.825332  0.821844
    RF+LR+SVM        stacking  meta=RF       0.825200  0.821844
RF

# Phase 4: Bootstrap CIs

In this phase, we perform a CiPA-style bootstrap validation of the single best configuration for each dataset at the `3class` scenario:
- **chantest**: stacking, base={RF, XGB}, meta=RandomForest
- **nanion**: stacking, base={RF, LR}, meta=RandomForest
- **manual**: stacking, base={XGB, LR, SVM}, meta=RandomForest

Each StackingClassifier is fit exactly once on the training split, and the test split (16 drugs, 2000 samples each) is resampled drug-level (exactly 16 rows, one drawn randomly per drug) for 10,000 iterations to compute medians and 95% confidence intervals (2.5th and 97.5th percentiles).

In [10]:
import sklearn
sklearn.set_config(enable_metadata_routing=True)

import time
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, recall_score, roc_auc_score

# Set seed for reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)

class FittedHolder:
    def __init__(self, estimator):
        self.estimator = estimator

class PreFittedEstimatorWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, holder=None):
        self.holder = holder
        if holder is not None:
            self.classes_ = holder.estimator.classes_
            
    def fit(self, X, y, sample_weight=None, groups=None):
        if self.holder is not None:
            self.classes_ = self.holder.estimator.classes_
        return self
        
    def predict(self, X):
        return self.holder.estimator.predict(X)
        
    def predict_proba(self, X):
        return self.holder.estimator.predict_proba(X)

class FinalEstimatorWrapper(ClassifierMixin, BaseEstimator):
    _estimator_type = 'classifier'
    def __init__(self, estimator=None):
        self.estimator = estimator
        if estimator is not None and hasattr(estimator, 'classes_'):
            self.classes_ = estimator.classes_
            
    def fit(self, X, y, sample_weight=None, groups=None, **kwargs):
        fit_params = {}
        if sample_weight is not None:
            fit_params['sample_weight'] = sample_weight
        self.estimator.fit(X, y, **fit_params)
        self.classes_ = self.estimator.classes_
        return self
        
    def predict(self, X):
        return self.estimator.predict(X)
        
    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

# Recover train groups
train_groups = {}
BIOMARKERS = ['qNet', 'dvdtmax', 'vmax', 'vrest', 'APD50', 'APD90', 'max_dv', 'camax', 'carest', 'CaTD50', 'CaTD90']
for ds in ['chantest', 'nanion', 'manual']:
    df_raw = datasets[f'{ds}_train']
    df_clean = df_raw.dropna(subset=BIOMARKERS)
    train_groups[ds] = df_clean['drug_name'].values

base_models_grouped = joblib.load('./artifacts/base_models_grouped.joblib')

# Define best configurations
configs_3class = {
    'chantest': {
        'base_names': ['RF', 'XGB'],
        'meta': RandomForestClassifier(random_state=SEED)
    },
    'nanion': {
        'base_names': ['RF', 'LR'],
        'meta': RandomForestClassifier(random_state=SEED)
    },
    'manual': {
        'base_names': ['XGB', 'LR', 'SVM'],
        'meta': RandomForestClassifier(random_state=SEED)
    }
}

bootstrap_results = {}
summary_table_rows = []

for dataset in ['chantest', 'nanion', 'manual']:
    start_time = time.time()
    print(f'Running Phase 4 Bootstrap for {dataset}...')
    
    # 1. Retrieve prep arrays
    bundle = prep[dataset]['3class']
    X_train = bundle['X_train']
    y_train = bundle['y_train']
    X_test = bundle['X_test']
    y_test = bundle['y_test']
    test_drugs = bundle['test_drugs']
    groups = train_groups[dataset]
    
    # 2. Rebuild winning StackingClassifier
    cfg = configs_3class[dataset]
    estimators = []
    for name in cfg['base_names']:
        base_est = base_models_grouped[dataset]['3class'][name]['estimator']
        w = PreFittedEstimatorWrapper(FittedHolder(base_est))
        w.set_fit_request(groups=True)
        estimators.append((name, w))
        
    final_est_wrapped = FinalEstimatorWrapper(cfg['meta'])
    
    cv = StratifiedGroupKFold(n_splits=4)
    cv.set_split_request(groups=True)
    
    model = StackingClassifier(
        estimators=estimators,
        final_estimator=final_est_wrapped,
        cv=cv,
        stack_method='predict_proba',
        passthrough=False
    )
    
    # 3. Fit exactly ONCE on Train Split
    model.fit(X_train, y_train, groups=groups)
    
    # 4. Predict ONCE on Full Test Split to optimize loop
    y_pred_full = model.predict(X_test)
    y_prob_full = model.predict_proba(X_test)
    
    # 5. Bootstrap Procedure (10,000 iterations)
    unique_drugs = np.unique(test_drugs)
    drug_indices = {d: np.where(test_drugs == d)[0] for d in unique_drugs}
    
    rng = np.random.RandomState(SEED)
    
    n_iterations = 10000
    f1_list = np.zeros(n_iterations)
    auc_high_list = np.zeros(n_iterations)
    auc_inter_list = np.zeros(n_iterations)
    auc_low_list = np.zeros(n_iterations)
    rec_high_list = np.zeros(n_iterations)
    rec_inter_list = np.zeros(n_iterations)
    rec_low_list = np.zeros(n_iterations)
    
    for i in range(n_iterations):
        # Drug-level resampling: draw one index per unique test drug
        sampled_indices = [rng.choice(drug_indices[d]) for d in unique_drugs]
        
        y_true_iter = y_test[sampled_indices]
        y_pred_iter = y_pred_full[sampled_indices]
        y_prob_iter = y_prob_full[sampled_indices]
        
        # Calculate macro-F1
        f1_list[i] = f1_score(y_true_iter, y_pred_iter, average='macro', zero_division=0)
        
        # Calculate per-class recall
        recalls = recall_score(y_true_iter, y_pred_iter, average=None, zero_division=0)
        rec_high_list[i] = recalls[0]
        rec_inter_list[i] = recalls[1]
        rec_low_list[i] = recalls[2]
        
        # Calculate one-vs-rest AUC per class
        # Class 0: high
        auc_high_list[i] = roc_auc_score((y_true_iter == 0).astype(int), y_prob_iter[:, 0])
        # Class 1: intermediate
        auc_inter_list[i] = roc_auc_score((y_true_iter == 1).astype(int), y_prob_iter[:, 1])
        # Class 2: low
        auc_low_list[i] = roc_auc_score((y_true_iter == 2).astype(int), y_prob_iter[:, 2])
        
    elapsed = time.time() - start_time
    print(f'bootstrap done: {dataset} | Runtime: {elapsed:.2f} seconds')
    
    # Store per-iteration arrays
    bootstrap_results[dataset] = {
        'f1': f1_list,
        'auc_high': auc_high_list,
        'auc_intermediate': auc_inter_list,
        'auc_low': auc_low_list,
        'recall_high': rec_high_list,
        'recall_intermediate': rec_inter_list,
        'recall_low': rec_low_list,
        'runtime_seconds': elapsed
    }
    
    # Compute median and 95% CIs
    metrics_info = {
        'macro-F1': f1_list,
        'AUC_high': auc_high_list,
        'AUC_intermediate': auc_inter_list,
        'AUC_low': auc_low_list,
        'recall_high': rec_high_list,
        'recall_intermediate': rec_inter_list,
        'recall_low': rec_low_list
    }
    
    dataset_rows = []
    for metric_name, arr in metrics_info.items():
        median = np.median(arr)
        ci_low = np.percentile(arr, 2.5)
        ci_high = np.percentile(arr, 97.5)
        dataset_rows.append({
            'dataset': dataset,
            'metric': metric_name,
            'median': median,
            'ci_low': ci_low,
            'ci_high': ci_high
        })
        
    bootstrap_results[dataset]['summary_table'] = pd.DataFrame(dataset_rows)
    summary_table_rows.extend(dataset_rows)

# Save results
joblib.dump(bootstrap_results, './artifacts/phase4_bootstrap.joblib')
summary_df = pd.DataFrame(summary_table_rows)
summary_df.to_csv('./artifacts/phase4_bootstrap_summary.csv', index=False)
print('\nBootstrap validation results saved successfully.')

# Print individual summary tables
for dataset in ['chantest', 'nanion', 'manual']:
    print(f'\nRESULTS TABLE FOR {dataset.upper()}:')
    print('='*60)
    ds_df = summary_df[summary_df['dataset'] == dataset][['metric', 'median', 'ci_low', 'ci_high']]
    print(ds_df.to_string(index=False))

# Combined Cross-dataset Summary Table
combined_rows = []
for dataset in ['chantest', 'nanion', 'manual']:
    ds_df = summary_df[summary_df['dataset'] == dataset].set_index('metric')
    
    def fmt_ci(metric_name):
        med = ds_df.loc[metric_name, 'median']
        low = ds_df.loc[metric_name, 'ci_low']
        high = ds_df.loc[metric_name, 'ci_high']
        return f'{med:.4f} [{low:.4f}, {high:.4f}]'
        
    combined_rows.append({
        'dataset': dataset,
        'macro-F1': fmt_ci('macro-F1'),
        'AUC_high': fmt_ci('AUC_high'),
        'AUC_intermediate': fmt_ci('AUC_intermediate'),
        'AUC_low': fmt_ci('AUC_low')
    })
    
combined_df = pd.DataFrame(combined_rows)
print('\n' + '='*80)
print('COMBINED CROSS-DATASET SUMMARY (MEDIAN [95% CI]):')
print('='*80)
print(combined_df.to_string(index=False))


Running Phase 4 Bootstrap for chantest...
bootstrap done: chantest | Runtime: 37.88 seconds
Running Phase 4 Bootstrap for nanion...
bootstrap done: nanion | Runtime: 60.49 seconds
Running Phase 4 Bootstrap for manual...
bootstrap done: manual | Runtime: 69.92 seconds

Bootstrap validation results saved successfully.

RESULTS TABLE FOR CHANTEST:
             metric   median   ci_low  ci_high
           macro-F1 0.819373 0.694017 0.878307
           AUC_high 0.916667 0.875000 0.958333
   AUC_intermediate 0.825397 0.714286 0.912698
            AUC_low 1.000000 0.836364 1.000000
        recall_high 1.000000 0.750000 1.000000
recall_intermediate 0.714286 0.428571 0.857143
         recall_low 1.000000 0.600000 1.000000

RESULTS TABLE FOR NANION:
             metric   median   ci_low  ci_high
           macro-F1 0.679365 0.623377 0.754762
           AUC_high 0.916667 0.875000 0.958333
   AUC_intermediate 0.658730 0.571429 0.746032
            AUC_low 0.700000 0.645455 0.800000
        recall_

# Phase 5a: Persist Deployable Model

In this phase, we rebuild and fit the deployable winner model (`chantest` × `3class` stacking ensemble with base={RF, XGB} and meta=RF) exactly once on the training set, bundle it with its scaler and metadata, and persist it to `./artifacts/deployable_chantest_3class.joblib`. We then verify the reload by asserting that reload predictions match the original predictions exactly (32,000/32,000) and demonstrating the unscaled prediction deployment path.

In [10]:
import sklearn
sklearn.set_config(enable_metadata_routing=True)

import time
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from deploy_utils import PreFittedEstimatorWrapper, FittedHolder, FinalEstimatorWrapper

# Set seed for reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)

# Recover train groups
train_groups = {}
BIOMARKERS = ['qNet', 'dvdtmax', 'vmax', 'vrest', 'APD50', 'APD90', 'max_dv', 'camax', 'carest', 'CaTD50', 'CaTD90']
for ds in ['chantest', 'nanion', 'manual']:
    df_raw = datasets[f'{ds}_train']
    df_clean = df_raw.dropna(subset=BIOMARKERS)
    train_groups[ds] = df_clean['drug_name'].values

base_models_grouped = joblib.load('./artifacts/base_models_grouped.joblib')

# Rebuild and fit the deployable model
print('Rebuilding deployable model for chantest x 3class...')
bundle = prep['chantest']['3class']
X_train = bundle['X_train']
y_train = bundle['y_train']
X_test = bundle['X_test']
y_test = bundle['y_test']
groups = train_groups['chantest']

estimators = []
for name in ['RF', 'XGB']:
    base_est = base_models_grouped['chantest']['3class'][name]['estimator']
    w = PreFittedEstimatorWrapper(FittedHolder(base_est))
    w.set_fit_request(groups=True)
    estimators.append((name, w))
    
final_est_wrapped = FinalEstimatorWrapper(RandomForestClassifier(random_state=SEED))

cv = StratifiedGroupKFold(n_splits=4)
cv.set_split_request(groups=True)

model = StackingClassifier(
    estimators=estimators,
    final_estimator=final_est_wrapped,
    cv=cv,
    stack_method='predict_proba',
    passthrough=False
)

model.fit(X_train, y_train, groups=groups)

# 1. Compute predictions before saving
preds_before = model.predict(X_test)
test_f1_before = f1_score(y_test, preds_before, average='macro', zero_division=0)
print(f'Macro-F1 before saving: {test_f1_before:.6f}')

# 2. Bundle it and save to disk
deployable = {
    'model': model,
    'scaler': bundle['scaler'],
    'biomarkers': BIOMARKERS,
    'class_map': {'high': 0, 'intermediate': 1, 'low': 2},
    'inverse_class_map': {0: 'high', 1: 'intermediate', 2: 'low'},
    'dataset': 'chantest',
    'scenario': '3class',
    'architecture': 'stacking base={RF,XGB} meta=RF',
    'cv_scheme': 'StratifiedGroupKFold(4) grouped by drug_name',
    'test_macro_f1_pointestimate': 0.828,
    'test_macro_f1_bootstrap_median_ci': '0.819 [0.694, 0.878]',
    'sklearn_version': sklearn.__version__,
    'seed': SEED
}

joblib.dump(deployable, './artifacts/deployable_chantest_3class.joblib')
print("Model bundle saved successfully to './artifacts/deployable_chantest_3class.joblib'.")

# Delete in-memory objects to prevent contamination
del deployable, model

# 3. Reload bundle from disk
reloaded_bundle = joblib.load('./artifacts/deployable_chantest_3class.joblib')
reloaded_model = reloaded_bundle['model']

# 4. Predict using reloaded model and verify
preds_after = reloaded_model.predict(X_test)
test_f1_after = f1_score(y_test, preds_after, average='macro', zero_division=0)

match_count = np.sum(preds_before == preds_after)
total_rows = len(X_test)
print(f'Predictions Match: {match_count}/{total_rows}')
print(f'Macro-F1 after reload: {test_f1_after:.6f}')

# Assert exact match
assert match_count == total_rows, 'ERROR: Reloaded predictions do not match original predictions!'
assert abs(test_f1_after - 0.828004) < 1e-4, f'ERROR: Reloaded model\'s F1 is {test_f1_after:.6f}, expected ~0.828'

# 5. Demonstrate intended deployment path on 3 unscaled example rows
print('\nDemonstrating deployment inference on 3 unscaled test rows:')
print('='*60)
df_test_raw = datasets['chantest_test']
df_test_clean = df_test_raw.dropna(subset=BIOMARKERS)

example_indices = []
for label_val in ['high', 'intermediate', 'low']:
    idx = df_test_clean[df_test_clean['label'] == label_val].index[0]
    clean_idx = df_test_clean.index.get_loc(idx)
    example_indices.append(clean_idx)

X_raw_examples = df_test_clean.iloc[example_indices][BIOMARKERS].values
y_true_labels = df_test_clean.iloc[example_indices]['label'].values

X_scaled_examples = reloaded_bundle['scaler'].transform(X_raw_examples)
y_pred_classes = reloaded_model.predict(X_scaled_examples)
y_pred_labels = [reloaded_bundle['inverse_class_map'][c] for c in y_pred_classes]

for i in range(3):
    print(f'Row {i+1}:')
    print(f'  Unscaled features: {X_raw_examples[i]}')
    print(f'  True Risk Label:   {y_true_labels[i]}')
    print(f'  Predicted Label:   {y_pred_labels[i]}')
    print(f'  Class match:       {y_true_labels[i] == y_pred_labels[i]}')


Rebuilding deployable model for chantest x 3class...
Macro-F1 before saving: 0.828004
Model bundle saved successfully to './artifacts/deployable_chantest_3class.joblib'.
Predictions Match: 32000/32000
Macro-F1 after reload: 0.828004

Demonstrating deployment inference on 3 unscaled test rows:
Row 1:
  Unscaled features: [ 4.36371180e+01  7.28793909e+01  4.18264540e+01 -8.80123071e+01
  3.38500000e+02  4.41250000e+02 -2.38296891e-01  2.54324000e-04
  7.60000000e-05  3.37500000e+02  7.67250000e+02]
  True Risk Label:   high
  Predicted Label:   high
  Class match:       True
Row 2:
  Unscaled features: [ 6.07198400e+01  7.29117096e+01  4.17932094e+01 -8.80141627e+01
  2.67250000e+02  3.28250000e+02 -3.29819743e-01  2.44544000e-04
  7.53000000e-05  3.11250000e+02  7.33500000e+02]
  True Risk Label:   intermediate
  Predicted Label:   intermediate
  Class match:       True
Row 3:
  Unscaled features: [ 6.96079540e+01  7.28962131e+01  4.16718326e+01 -8.80143740e+01
  2.38000000e+02  2.90000

# Phase 5b: SHAP Explainability

In this phase, we perform a black-box SHAP explainability analysis on the reloaded deployable model (`deployable_chantest_3class.joblib`).
Because this is a 3-class stacking ensemble, we wrap the pipeline (scaler transform + model predict_proba) and explain it as a black-box using `shap.KernelExplainer` to attribute importances directly to the 11 original biomarkers. We use a background sample of 100 scaled training rows and an evaluation sample of 500 scaled test rows (seed=42 for both). We plot beeswarm plots per class and save the SHAP values.

### Interpretation Note:
The top biomarkers identified overall are:
1. **qNet**: The net charge metric, which is the primary FDA-proposed biomarker in the CiPA initiative for integrating multi-channel blockade into a single arrhythmogenic risk metric.
2. **vrest**: Resting membrane potential, which affects sodium channel availability.
3. **APD50** & **APD90**: Action potential duration metrics, which are directly related to prolongation of cardiac repolarization (the cellular basis of long QT syndrome and TdP risk).
These align perfectly with the known drivers of TdP risk in the clinical and physiological cardiac literature.

In [10]:
import joblib
import numpy as np
import pandas as pd
import time
import os
import shap
import matplotlib.pyplot as plt
from deploy_utils import PreFittedEstimatorWrapper, FittedHolder, FinalEstimatorWrapper

# Set seed for reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)

# Load deployable bundle
deployable = joblib.load('./artifacts/deployable_chantest_3class.joblib')
model = deployable['model']
scaler = deployable['scaler']
biomarkers = deployable['biomarkers']
class_map = deployable['class_map']
inverse_class_map = deployable['inverse_class_map']

# Load raw datasets
data_dir = './data'
df_train_raw = pd.read_csv(os.path.join(data_dir, 'chantest_training.csv'))
df_test_raw = pd.read_csv(os.path.join(data_dir, 'chantest_testing.csv'))

df_train_clean = df_train_raw.dropna(subset=biomarkers)
df_test_clean = df_test_raw.dropna(subset=biomarkers)

X_train_raw = df_train_clean[biomarkers].values
X_test_raw = df_test_clean[biomarkers].values

X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Choose background sample (100 rows, seed=42)
rng = np.random.RandomState(SEED)
bg_indices = rng.choice(X_train_scaled.shape[0], size=100, replace=False)
background_sample = X_train_scaled[bg_indices]

# Choose evaluation sample (500 rows, seed=42)
eval_indices = rng.choice(X_test_scaled.shape[0], size=500, replace=False)
eval_sample = X_test_scaled[eval_indices]

print('Background sample shape:', background_sample.shape)
print('Eval sample shape:', eval_sample.shape)

# Run KernelExplainer
start_time = time.time()

def predict_proba_wrapper(X):
    return model.predict_proba(X)

explainer = shap.KernelExplainer(predict_proba_wrapper, background_sample)
print('Computing SHAP values for 500 rows (nsamples=200)...')
shap_values = explainer.shap_values(eval_sample, nsamples=200)
elapsed = time.time() - start_time
print(f'SHAP computation done in {elapsed:.2f} seconds.')

# Save plots
os.makedirs('./artifacts/shap', exist_ok=True)
for c_idx, c_name in [(0, 'high'), (1, 'intermediate'), (2, 'low')]:
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values[:, :, c_idx], eval_sample, feature_names=biomarkers, show=False)
    plt.title(f'SHAP Summary (Beeswarm) Plot - {c_name.upper()} Class')
    plt.tight_layout()
    plt.savefig(f'./artifacts/shap/shap_summary_{c_name}.png', dpi=150)
    plt.close()
print('SHAP beeswarm plots saved successfully.')

# Compute global feature importances
global_importances = np.mean(np.abs(shap_values), axis=(0, 2))
global_df = pd.DataFrame({
    'biomarker': biomarkers,
    'mean_abs_shap': global_importances
}).sort_values(by='mean_abs_shap', ascending=False)

print('\nGLOBAL FEATURE IMPORTANCE:')
print(global_df.to_string(index=False))

# Per-class importances
per_class_importances = {}
for c_idx, c_name in [(0, 'high'), (1, 'intermediate'), (2, 'low')]:
    class_imp = np.mean(np.abs(shap_values[:, :, c_idx]), axis=0)
    class_df = pd.DataFrame({
        'biomarker': biomarkers,
        'mean_abs_shap': class_imp
    }).sort_values(by='mean_abs_shap', ascending=False)
    per_class_importances[c_name] = class_df
    print(f'\nIMPORTANCE FOR {c_name.upper()} CLASS:')
    print(class_df.to_string(index=False))

# Save raw SHAP values and tables
shap_data = {
    'shap_values': shap_values,
    'eval_sample': eval_sample,
    'biomarkers': biomarkers,
    'global_importance_table': global_df,
    'per_class_importance_tables': per_class_importances
}
joblib.dump(shap_data, './artifacts/shap_values.joblib')
print('\nSHAP values and importance tables saved to `./artifacts/shap_values.joblib`.')


Background sample shape: (100, 11)
Eval sample shape: (500, 11)
Computing SHAP values for 500 rows (nsamples=200)...
SHAP computation done in 167.82 seconds.
SHAP beeswarm plots saved successfully.

GLOBAL FEATURE IMPORTANCE:
biomarker  mean_abs_shap
     qNet       0.161796
    vrest       0.083903
    APD50       0.071386
   max_dv       0.062798
    APD90       0.058283
     vmax       0.030772
    camax       0.017091
   CaTD50       0.011256
   CaTD90       0.010575
  dvdtmax       0.007295
   carest       0.005708

IMPORTANCE FOR HIGH CLASS:
biomarker  mean_abs_shap
     qNet       0.160063
   max_dv       0.090272
    APD90       0.077279
    APD50       0.056726
    vrest       0.023126
    camax       0.019041
   CaTD90       0.011334
     vmax       0.010201
   CaTD50       0.006984
  dvdtmax       0.006540
   carest       0.005448

IMPORTANCE FOR INTERMEDIATE CLASS:
biomarker  mean_abs_shap
    vrest       0.122554
     qNet       0.118141
   max_dv       0.081828
    APD90 

100%|##########| 500/500 [02:47<00:00,  2.98it/s]
<string>:66: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
<string>:66: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
<string>:66: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.


# Phase 5c: Thesis Figures

In this phase, we generate the final thesis-quality figures from our frozen evaluation artifacts, saving them to `./artifacts/figures/` at 300 DPI with tight bounding boxes and clear labels:
1. `confusion_matrix_chantest.png`: Confusion matrix of the deployable model on the test split, showing raw counts and normalized percentages.
2. `bootstrap_macrof1_distributions.png`: KDE plots of the 10,000 bootstrap macro-F1 values comparing chantest, nanion, and manual datasets.
3. `per_class_auc_comparison.png`: Grouped bar chart of bootstrap AUC values with 95% confidence interval error bars.
4. `shap_global_importance.png`: Horizontal bar chart of mean absolute SHAP values across all biomarkers.
5. `ensemble_family_comparison.png`: Bar plots showing performance comparing Stacking vs Voting, and pure-ML vs pure-DL vs mixed paradigms.

In [10]:
import time
import os
import shutil
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score
from deploy_utils import PreFittedEstimatorWrapper, FittedHolder, FinalEstimatorWrapper

# Set seaborn style for clean, beautiful plots
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans', 'sans-serif'],
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 0.8,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'text.color': '#111111',
    'axes.labelcolor': '#111111',
    'axes.titlecolor': '#111111',
    'figure.titlesize': 18,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10
})

os.makedirs('./artifacts/figures', exist_ok=True)

# Load frozen artifacts
deployable = joblib.load('./artifacts/deployable_chantest_3class.joblib')
reloaded_model = deployable['model']
scaler = deployable['scaler']
biomarkers = deployable['biomarkers']
inverse_class_map = deployable['inverse_class_map']

bootstrap = joblib.load('./artifacts/phase4_bootstrap.joblib')
shap_data = joblib.load('./artifacts/shap_values.joblib')
sweep_results = pd.read_csv('./artifacts/phase3_protocolA_results.csv')

# Retrieve test split for chantest
bundle = prep['chantest']['3class']
X_test = bundle['X_test']
y_test = bundle['y_test']

# 1. confusion_matrix_chantest.png
print('Generating Figure 1: Confusion Matrix...')
preds = reloaded_model.predict(X_test)
cm = confusion_matrix(y_test, preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

labels = ['high', 'intermediate', 'low']
plt.figure(figsize=(7, 6))
sns.heatmap(cm_normalized, annot=False, cmap='Blues', cbar=True,
            xticklabels=labels, yticklabels=labels, vmin=0, vmax=1)

# Annotate with count and percentage
for i in range(3):
    for j in range(3):
        color = 'white' if cm_normalized[i, j] > 0.5 else 'black'
        plt.text(j + 0.5, i + 0.5, f'{cm[i, j]:,}\n({cm_normalized[i, j]*100:.1f}%)',
                 horizontalalignment='center', verticalalignment='center',
                 color=color, fontsize=12, weight='bold')

plt.ylabel('True Risk Category', fontsize=12, fontweight='bold', labelpad=10)
plt.xlabel('Predicted Risk Category', fontsize=12, fontweight='bold', labelpad=10)
plt.title('Confusion Matrix: Deployable Stacking Model on Test Split\n(Chantest 3-Class, Test Macro-F1 = 0.828)', 
          fontsize=12, pad=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./artifacts/figures/confusion_matrix_chantest.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. bootstrap_macrof1_distributions.png
print('Generating Figure 2: Bootstrap Macro-F1 Distributions...')
plt.figure(figsize=(9, 5))
colors = {'chantest': '#1f77b4', 'nanion': '#ff7f0e', 'manual': '#2ca02c'}

for ds in ['chantest', 'nanion', 'manual']:
    f1_arr = bootstrap[ds]['f1']
    median = np.median(f1_arr)
    ci_low = np.percentile(f1_arr, 2.5)
    ci_high = np.percentile(f1_arr, 97.5)
    
    sns.kdeplot(f1_arr, label=f'{ds} (median={median:.3f})', color=colors[ds], fill=True, alpha=0.25, linewidth=2)
    plt.axvline(median, color=colors[ds], linestyle='-', linewidth=1.5)
    plt.axvline(ci_low, color=colors[ds], linestyle='--', linewidth=1)
    plt.axvline(ci_high, color=colors[ds], linestyle='--', linewidth=1)

plt.xlabel('Bootstrap Test Macro-F1', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Bootstrap Test Macro-F1 Distributions (10,000 Iterations)\nSolid lines = Median, Dashed lines = 95% CI', fontsize=12, pad=15, fontweight='bold')
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig('./artifacts/figures/bootstrap_macrof1_distributions.png', dpi=300, bbox_inches='tight')
plt.close()

# 3. per_class_auc_comparison.png
print('Generating Figure 3: Per-class AUC Comparison...')
datasets_list = ['chantest', 'nanion', 'manual']
classes = ['high', 'intermediate', 'low']
x_indices = np.arange(len(datasets_list))
bar_width = 0.25

plt.figure(figsize=(9, 6))

for c_idx, c_name in enumerate(classes):
    medians = []
    yerr_low = []
    yerr_high = []
    
    for ds in datasets_list:
        auc_arr = bootstrap[ds][f'auc_{c_name}']
        med = np.median(auc_arr)
        low = np.percentile(auc_arr, 2.5)
        high = np.percentile(auc_arr, 97.5)
        
        medians.append(med)
        yerr_low.append(med - low)
        yerr_high.append(high - med)
        
    yerr = np.array([yerr_low, yerr_high])
    plt.bar(x_indices + c_idx * bar_width, medians, bar_width, 
            yerr=yerr, capsize=4, label=f'{c_name.upper()} Class', 
            error_kw=dict(lw=1.2, capthick=1.2, ecolor='#333333'))

plt.xticks(x_indices + bar_width, [d.capitalize() for d in datasets_list], fontsize=11)
plt.ylabel('One-vs-Rest Area Under ROC (AUC)', fontsize=12)
plt.xlabel('Dataset Split', fontsize=12)
plt.title('Per-Class AUC Comparison with 95% Bootstrap Confidence Intervals', fontsize=12, pad=15, fontweight='bold')
plt.ylim(0.5, 1.05)
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig('./artifacts/figures/per_class_auc_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

# 4. shap_global_importance.png
print('Generating Figure 4: SHAP Global Feature Importance...')
global_df = shap_data['global_importance_table'].sort_values(by='mean_abs_shap', ascending=True)

plt.figure(figsize=(9, 6))
sns.barplot(data=global_df, x='mean_abs_shap', y='biomarker', color='#2ca02c', edgecolor='#1a651a')
plt.xlabel('Mean Absolute SHAP Value (aggregated across classes)', fontsize=12)
plt.ylabel('CiPA Biomarker', fontsize=12)
plt.title('Global Feature Importance of Reloaded Deployable Model (Chantest 3-Class)', fontsize=12, pad=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./artifacts/figures/shap_global_importance.png', dpi=300, bbox_inches='tight')
plt.close()

# 5. ensemble_family_comparison.png
print('Generating Figure 5: Ensemble Family & Paradigm Comparison...')
sweep_c3 = sweep_results[(sweep_results['dataset'] == 'chantest') & (sweep_results['scenario'] == '3class')]

best_stacking = sweep_c3[sweep_c3['ensemble_family'] == 'stacking']['test_macro_f1'].max()
best_voting = sweep_c3[sweep_c3['ensemble_family'] == 'voting']['test_macro_f1'].max()

best_ml = sweep_c3[sweep_c3['paradigm'] == 'ML']['test_macro_f1'].max()
best_dl = sweep_c3[sweep_c3['paradigm'] == 'DL']['test_macro_f1'].max()
best_mixed = sweep_c3[sweep_c3['paradigm'] == 'mixed']['test_macro_f1'].max()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.bar(['Stacking', 'Voting'], [best_stacking, best_voting], color=['#1f77b4', '#aec7e8'], edgecolor='#111111', width=0.5)
ax1.set_ylabel('Best Test Macro-F1', fontsize=11)
ax1.set_ylim(0.7, 0.85)
ax1.set_title('Stacking vs. Voting Ensembles', fontsize=11, fontweight='bold')

ax2.bar(['ML', 'DL (ANNs)', 'Mixed'], [best_ml, best_dl, best_mixed], color=['#2ca02c', '#d62728', '#9467bd'], edgecolor='#111111', width=0.6)
ax2.set_ylabel('Best Test Macro-F1', fontsize=11)
ax2.set_ylim(0.2, 0.85)
ax2.set_title('Pure-ML vs. Pure-DL vs. Mixed', fontsize=11, fontweight='bold')

plt.suptitle('Combinatorial Ensemble Sweep Comparison (Chantest 3-Class)', fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('./artifacts/figures/ensemble_family_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

# 6. Copy the three SHAP beeswarm summary plots
print('Copying existing SHAP beeswarm summary plots...')
for c_name in ['high', 'intermediate', 'low']:
    src = f'./artifacts/shap/shap_summary_{c_name}.png'
    dst = f'./artifacts/figures/shap_summary_{c_name}.png'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied {src} to {dst}')
    else:
        print(f'WARNING: Source SHAP plot not found: {src}')

print('\nAll figures generated and saved successfully to \'./artifacts/figures/\'.')

print('\nFiles in \'./artifacts/figures/\':')
print('='*60)
for filename in sorted(os.listdir('./artifacts/figures')):
    print(f'  - {filename}')


Generating Figure 1: Confusion Matrix...
Generating Figure 2: Bootstrap Macro-F1 Distributions...
Generating Figure 3: Per-class AUC Comparison...
Generating Figure 4: SHAP Global Feature Importance...
Generating Figure 5: Ensemble Family & Paradigm Comparison...
Copying existing SHAP beeswarm summary plots...
Copied ./artifacts/shap/shap_summary_high.png to ./artifacts/figures/shap_summary_high.png
Copied ./artifacts/shap/shap_summary_intermediate.png to ./artifacts/figures/shap_summary_intermediate.png
Copied ./artifacts/shap/shap_summary_low.png to ./artifacts/figures/shap_summary_low.png

All figures generated and saved successfully to './artifacts/figures/'.

Files in './artifacts/figures/':
  - bootstrap_macrof1_distributions.png
  - confusion_matrix_chantest.png
  - ensemble_family_comparison.png
  - per_class_auc_comparison.png
  - shap_global_importance.png
  - shap_summary_high.png
  - shap_summary_intermediate.png
  - shap_summary_low.png
